# Exercise 3 — MACD

MACD (Moving Average Convergence/Divergence) shows the relationship between two EMAs. The MACD line crosses above the signal line as a bullish signal; crossing below is bearish. The histogram shows the gap between the two lines — its direction often signals momentum shifts before the crossover happens.

In [ ]:
import pandas as pd, math

def _synthetic(n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def sma(series, window=20):
    return series.rolling(window=window).mean()
def ema(series, window=20):
    return series.ewm(span=window, adjust=False).mean()
def rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(window=window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window=window).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))

# ── Exercise: implement macd ──────────────────────────────────────────────────

def macd(series, fast=12, slow=26, signal=9):
    """MACD: Moving Average Convergence/Divergence.

    Args:
        series : pd.Series of prices
        fast   : fast EMA window (default 12)
        slow   : slow EMA window (default 26)
        signal : signal-line EMA applied to the MACD line (default 9)

    Returns:
        pd.DataFrame with columns:
            macd      — ema(series, fast) - ema(series, slow)
            signal    — ema(macd_line, signal)
            histogram — macd_line - signal_line
    """
    # TODO:
    # 1. fast_ema  = ema(series, fast)
    # 2. slow_ema  = ema(series, slow)
    # 3. macd_line = fast_ema - slow_ema
    # 4. signal_line = ema(macd_line, signal)
    # 5. histogram = macd_line - signal_line
    # 6. return pd.DataFrame({"macd": macd_line, "signal": signal_line, "histogram": histogram})
    n = len(series)
    return pd.DataFrame({"macd":      [0.0] * n,
                          "signal":    [0.0] * n,
                          "histogram": [0.0] * n},
                         index=series.index)


### Checks

In [ ]:
checks = 0

# 1 — macd returns DataFrame with 3 expected columns
try:
    close = _synthetic()["Close"]
    m = macd(close)
    assert isinstance(m, pd.DataFrame)
    assert "macd" in m.columns and "signal" in m.columns and "histogram" in m.columns
    assert len(m) == len(close)
    checks += 1; print("✅ 1 macd returns DataFrame with macd/signal/histogram columns")
except Exception as e:
    print("❌ 1:", e)

# 2 — histogram = macd - signal for every row
try:
    close = _synthetic()["Close"]
    m = macd(close)
    diff = (m["macd"] - m["signal"] - m["histogram"]).abs().max()
    assert diff < 1e-9, f"histogram != macd - signal, max diff={diff}"
    checks += 1; print("✅ 2 histogram == macd - signal for every row")
except Exception as e:
    print("❌ 2:", e)

# 3 — no NaN values (EMA starts from first observation)
try:
    close = _synthetic()["Close"]
    m = macd(close)
    assert not m.isna().any().any(), f"unexpected NaN in MACD DataFrame"
    checks += 1; print("✅ 3 MACD has no NaN values")
except Exception as e:
    print("❌ 3:", e)

# 4 — macd of a constant series is zero everywhere
try:
    const = pd.Series([50.0] * 50)
    m = macd(const)
    assert m["macd"].abs().max() < 1e-9, f"MACD of constant should be 0, got {m['macd'].abs().max()}"
    checks += 1; print("✅ 4 macd of constant price series is zero")
except Exception as e:
    print("❌ 4:", e)

# 5 — macd line is positive when fast EMA > slow EMA
try:
    # Linearly rising price: fast EMA tracks faster -> fast > slow -> MACD > 0
    rising = pd.Series([float(i) for i in range(1, 51)])
    m = macd(rising, fast=5, slow=20, signal=3)
    # After warmup (say, last 20 rows), MACD should be positive
    last20 = m["macd"].iloc[-20:]
    assert (last20 > 0).all(), f"expected MACD > 0 for rising prices: {last20.tolist()}"
    checks += 1; print("✅ 5 MACD is positive when price has been rising steadily")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
